<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Air-Quality-_PM-2.5/Air_Quality_PM_2_5_3_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load your cleaned final PM2.5 dataset
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/final_pm25_dataset.csv"
pm25_df = pd.read_csv(path)

print("Dataset shape:", pm25_df.shape)
pm25_df.head()

Dataset shape: (130003, 16)


,air_quality_PM10,air_quality_Carbon_Monoxide,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,humidity,cloud,visibility_km,longitude,temperature_celsius,condition_text,uv_index,wind_mph,precip_mm,gust_mph,wind_degree,air_quality_PM2.5
0,7.1,198.6,2.5,0.2,58,0,16.0,-120.49,16.1,2,1.0,4.3,0.00,10.3,220,6.3
1,25.3,377.2,3.7,1.4,78,37,10.0,-87.22,23.0,32,1.0,3.8,0.28,7.0,240,19.0
2,28.1,460.6,7.7,7.5,94,50,10.0,-89.20,26.0,23,1.0,2.2,0.30,2.8,182,20.4
3,178.1,2243.0,35.0,19.3,88,100,5.0,-90.53,20.0,19,1.0,13.6,0.09,18.1,190,132.0
4,32.1,307.1,0.3,0.2,89,94,10.0,-88.77,26.0,30,1.0,4.3,0.00,6.5,99,7.7


In [ ]:
target = "air_quality_PM2.5"

selected_features = [col for col in pm25_df.columns if col != target]

print("Target:", target)
print("Number of features:", len(selected_features))
print(selected_features)

Target: air_quality_PM2.5
Number of features: 15
['air_quality_PM10', 'air_quality_Carbon_Monoxide', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'humidity', 'cloud', 'visibility_km', 'longitude', 'temperature_celsius', 'condition_text', 'uv_index', 'wind_mph', 'precip_mm', 'gust_mph', 'wind_degree']


In [ ]:
split_index = int(0.8 * len(pm25_df))

train_df = pm25_df.iloc[:split_index].copy()
test_df = pm25_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (104002, 16)
Test shape : (26001, 16)


In [ ]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34320, 16)
Iteration 2 shape: (34321, 16)
Iteration 3 shape: (35361, 16)


In [ ]:
def get_xy(data):
    X = data[selected_features]
    y = data[target]
    return X, y

def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    accuracy = r2 * 100
    return mse, rmse, mae, r2, accuracy

In [ ]:
X1, y1 = get_xy(iter1)
X2, y2 = get_xy(iter2)
X3, y3 = get_xy(iter3)

X_test, y_test = get_xy(test_df)

print(X1.shape, X2.shape, X3.shape, X_test.shape)

(34320, 15) (34321, 15) (35361, 15) (26001, 15)


In [ ]:
results = []            # for iteration-wise RMSE comparison table
all_iteration_metrics = []  # for full training iteration metrics of each model
final_test_results = [] # for final unseen test results

# **SGD Regressor**

In [ ]:
from sklearn.linear_model import SGDRegressor

scaler_sgd = StandardScaler()

X1_sgd = scaler_sgd.fit_transform(X1)
X2_sgd = scaler_sgd.transform(X2)
X3_sgd = scaler_sgd.transform(X3)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = SGDRegressor(random_state=42)

# Iteration 1
sgd_model.partial_fit(X1_sgd, y1)
sgd_pred1 = sgd_model.predict(X1_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y1, sgd_pred1)
results.append(["Iteration 1", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 1", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Iteration 2
sgd_model.partial_fit(X2_sgd, y2)
sgd_pred2 = sgd_model.predict(X2_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y2, sgd_pred2)
results.append(["Iteration 2", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 2", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Iteration 3
sgd_model.partial_fit(X3_sgd, y3)
sgd_pred3 = sgd_model.predict(X3_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y3, sgd_pred3)
results.append(["Iteration 3", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 3", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Final test
sgd_test_pred = sgd_model.predict(X_test_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, sgd_test_pred)
final_test_results.append(["Linear (SGD)", mse, rmse, mae, r2, acc])

# **CNN**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

scaler_cnn = StandardScaler()

X1_cnn = scaler_cnn.fit_transform(X1)
X2_cnn = scaler_cnn.transform(X2)
X3_cnn = scaler_cnn.transform(X3)
X_test_cnn = scaler_cnn.transform(X_test)

X1_cnn = X1_cnn.reshape((X1_cnn.shape[0], X1_cnn.shape[1], 1))
X2_cnn = X2_cnn.reshape((X2_cnn.shape[0], X2_cnn.shape[1], 1))
X3_cnn = X3_cnn.reshape((X3_cnn.shape[0], X3_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X1_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu', padding='same'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

# Iteration 1
cnn_model.fit(X1_cnn, y1, epochs=10, batch_size=32, verbose=0)
cnn_pred1 = cnn_model.predict(X1_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y1, cnn_pred1)
results.append(["Iteration 1", "CNN", rmse])
all_iteration_metrics.append(["Iteration 1", "CNN", mse, rmse, mae, r2, acc])

# Iteration 2
cnn_model.fit(X2_cnn, y2, epochs=10, batch_size=32, verbose=0)
cnn_pred2 = cnn_model.predict(X2_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y2, cnn_pred2)
results.append(["Iteration 2", "CNN", rmse])
all_iteration_metrics.append(["Iteration 2", "CNN", mse, rmse, mae, r2, acc])

# Iteration 3
cnn_model.fit(X3_cnn, y3, epochs=10, batch_size=32, verbose=0)
cnn_pred3 = cnn_model.predict(X3_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y3, cnn_pred3)
results.append(["Iteration 3", "CNN", rmse])
all_iteration_metrics.append(["Iteration 3", "CNN", mse, rmse, mae, r2, acc])

# Final test
cnn_test_pred = cnn_model.predict(X_test_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y_test, cnn_test_pred)
final_test_results.append(["CNN", mse, rmse, mae, r2, acc])

# **XG Boost**

In [ ]:
# !pip install xgboost

import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

# Iteration 1
xgb_model.fit(X1, y1)
xgb_pred1 = xgb_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, xgb_pred1)
results.append(["Iteration 1", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 1", "XGBoost", mse, rmse, mae, r2, acc])

# Iteration 2
xgb_model.fit(X2, y2, xgb_model=xgb_model.get_booster())
xgb_pred2 = xgb_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, xgb_pred2)
results.append(["Iteration 2", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 2", "XGBoost", mse, rmse, mae, r2, acc])

# Iteration 3
xgb_model.fit(X3, y3, xgb_model=xgb_model.get_booster())
xgb_pred3 = xgb_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, xgb_pred3)
results.append(["Iteration 3", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 3", "XGBoost", mse, rmse, mae, r2, acc])

# Final test
xgb_test_pred = xgb_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, xgb_test_pred)
final_test_results.append(["XGBoost", mse, rmse, mae, r2, acc])

# Random **Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Iteration 1
rf_model.fit(X1, y1)
rf_pred1 = rf_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, rf_pred1)
results.append(["Iteration 1", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 1", "Random Forest", mse, rmse, mae, r2, acc])

# Iteration 2
X12 = pd.concat([X1, X2])
y12 = pd.concat([y1, y2])
rf_model.fit(X12, y12)
rf_pred2 = rf_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, rf_pred2)
results.append(["Iteration 2", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 2", "Random Forest", mse, rmse, mae, r2, acc])

# Iteration 3
X123 = pd.concat([X1, X2, X3])
y123 = pd.concat([y1, y2, y3])
rf_model.fit(X123, y123)
rf_pred3 = rf_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, rf_pred3)
results.append(["Iteration 3", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 3", "Random Forest", mse, rmse, mae, r2, acc])

# Final test
rf_test_pred = rf_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, rf_test_pred)
final_test_results.append(["Random Forest", mse, rmse, mae, r2, acc])

# #Light **GBM**

In [ ]:
# !pip install lightgbm

import lightgbm as lgb

lgb_train1 = lgb.Dataset(X1, label=y1)
lgb_train2 = lgb.Dataset(X2, label=y2)
lgb_train3 = lgb.Dataset(X3, label=y3)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "verbose": -1
}

# Iteration 1
lgb_model = lgb.train(params, lgb_train1, num_boost_round=100)
lgb_pred1 = lgb_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, lgb_pred1)
results.append(["Iteration 1", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 1", "LightGBM", mse, rmse, mae, r2, acc])

# Iteration 2
lgb_model = lgb.train(params, lgb_train2, num_boost_round=100, init_model=lgb_model)
lgb_pred2 = lgb_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, lgb_pred2)
results.append(["Iteration 2", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 2", "LightGBM", mse, rmse, mae, r2, acc])

# Iteration 3
lgb_model = lgb.train(params, lgb_train3, num_boost_round=100, init_model=lgb_model)
lgb_pred3 = lgb_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, lgb_pred3)
results.append(["Iteration 3", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 3", "LightGBM", mse, rmse, mae, r2, acc])

# Final test
lgb_test_pred = lgb_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, lgb_test_pred)
final_test_results.append(["LightGBM", mse, rmse, mae, r2, acc])

In [ ]:
all_iteration_metrics_df = pd.DataFrame(
    all_iteration_metrics,
    columns=["Iteration", "Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

all_iteration_metrics_df

,Iteration,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Iteration 1,Linear (SGD),2772.489,52.654,21.492,-0.275,-27.544
1,Iteration 2,Linear (SGD),436.504,20.893,12.579,0.709,70.914
2,Iteration 3,Linear (SGD),187.967,13.710,8.131,0.817,81.717
3,Iteration 1,CNN,94.400,9.716,3.809,0.957,95.657
4,Iteration 2,CNN,95.267,9.760,5.502,0.937,93.652
5,Iteration 3,CNN,37.729,6.142,3.591,0.963,96.330
6,Iteration 1,XGBoost,25.241,5.024,2.526,0.988,98.839
7,Iteration 2,XGBoost,115.182,10.732,4.374,0.923,92.325
8,Iteration 3,XGBoost,156.696,12.518,3.621,0.848,84.759
9,Iteration 1,Random Forest,4.551,2.133,0.840,0.998,99.791


# **Three Iterations Results_ Training**

In [ ]:
results_df = pd.DataFrame(results, columns=["Iteration", "Model", "RMSE"])

comparison_table = results_df.pivot(
    index="Iteration",
    columns="Model",
    values="RMSE"
)

comparison_table = comparison_table.reindex(["Iteration 1", "Iteration 2", "Iteration 3"]).round(3)

comparison_table

Model,CNN,LightGBM,Linear (SGD),Random Forest,XGBoost
Iteration,,,,,
Iteration 1,9.716,7.255,52.654,2.133,5.024
Iteration 2,9.760,7.844,20.893,3.234,10.732
Iteration 3,6.142,5.410,13.710,2.147,12.518


# **Test Results of all**

In [ ]:
final_test_df = pd.DataFrame(
    final_test_results,
    columns=["Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

final_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),198.464,14.088,8.343,0.669,66.851
1,CNN,58.178,7.627,3.601,0.903,90.283
2,XGBoost,188.405,13.726,3.643,0.685,68.531
3,Random Forest,51.866,7.202,3.330,0.913,91.337
4,LightGBM,101.572,10.078,3.170,0.830,83.035


In [ ]:
from google.colab import files

all_iteration_metrics_df.to_csv("pm25_training_iteration_metrics_all_models.csv", index=False)
comparison_table.to_csv("pm25_iteration_comparison_table.csv")
final_test_df.to_csv("pm25_final_test_results.csv", index=False)

files.download("pm25_training_iteration_metrics_all_models.csv")
files.download("pm25_iteration_comparison_table.csv")
files.download("pm25_final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Training Dataset**
# **Testing Dataset**

In [ ]:
train_df.to_csv("pm25_training_dataset.csv", index=False)
test_df.to_csv("pm25_testing_dataset.csv", index=False)

from google.colab import files
files.download("pm25_training_dataset.csv")
files.download("pm25_testing_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>